In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize_scalar
import json
import pickle
from xgboost import XGBRegressor

In [6]:
# Load XGBoost models
home_model = XGBRegressor()
home_model.load_model("../models/home_goals_model.json")

away_model = XGBRegressor()
away_model.load_model("../models/away_goals_model.json")

# Load Elo history
elo_history = pd.read_csv("../data/processed/elo_history.csv")
elo_history["date"] = pd.to_datetime(elo_history["date"])
# Load results
results = pd.read_csv("../data/raw/kaggle/results.csv")
results["date"] = pd.to_datetime(results["date"])

In [7]:
CALIB_END = "2018-06-14"  # WC 2018 start date

calib = results[
    (results["date"] < CALIB_END) &
    (results["tournament"] != "Friendly")
].copy()

print(f"Calibration matches: {len(calib)}")

Calibration matches: 25277


In [9]:
# Build a flat team-date-elo table first (do this once)
home_elo_df = elo_history[["date", "home_team", "home_elo"]].rename(
    columns={"home_team": "team", "home_elo": "elo"}
)
away_elo_df = elo_history[["date", "away_team", "away_elo"]].rename(
    columns={"away_team": "team", "away_elo": "elo"}
)
team_elo_df = pd.concat([home_elo_df, away_elo_df]).sort_values("date").reset_index(drop=True)

# For each team, get their most recent Elo before each date via merge_asof
calib_sorted = calib.sort_values("date").reset_index(drop=True)

# Home Elo
home_lookup = team_elo_df.rename(columns={"team": "home_team", "elo": "home_elo"})
calib_sorted = pd.merge_asof(calib_sorted, home_lookup, on="date", by="home_team")

# Away Elo
away_lookup = team_elo_df.rename(columns={"team": "away_team", "elo": "away_elo"})
calib_sorted = pd.merge_asof(calib_sorted, away_lookup, on="date", by="away_team")

calib_sorted["elo_diff"] = calib_sorted["home_elo"] - calib_sorted["away_elo"]
calib_sorted["is_worldcup"] = calib_sorted["tournament"].str.contains("World Cup", na=False).astype(int)
calib_sorted[["home_elo", "away_elo"]] = calib_sorted[["home_elo", "away_elo"]].fillna(1500)

calib_features = calib_sorted[["home_elo", "away_elo", "elo_diff", "is_worldcup", "home_score", "away_score"]].copy()

print(calib_features.shape)
print(calib_features.head())

(25277, 6)
      home_elo     away_elo    elo_diff  is_worldcup  home_score  away_score
0  1446.929947  1649.541147 -202.611199            0         0.0         5.0
1  1426.943318  1439.329304  -12.385986            0         6.0         0.0
2  1422.759150  1476.585588  -53.826438            0         1.0         8.0
3  1657.141790  1490.126432  167.015359            0         1.0         0.0
4  1443.513472  1481.275410  -37.761937            0         0.0         4.0


In [10]:
feature_cols = ["home_elo", "away_elo", "elo_diff", "is_worldcup"]

calib_features["pred_home"] = home_model.predict(calib_features[feature_cols])
calib_features["pred_away"] = away_model.predict(calib_features[feature_cols])

# Clip to avoid negative lambdas
calib_features["pred_home"] = calib_features["pred_home"].clip(lower=0.1)
calib_features["pred_away"] = calib_features["pred_away"].clip(lower=0.1)

print(calib_features[["pred_home", "pred_away"]].describe())

          pred_home     pred_away
count  25277.000000  25277.000000
mean       1.671961      1.164456
std        0.835652      0.601291
min        0.181813      0.160372
25%        1.059604      0.731163
50%        1.520786      1.048139
75%        2.121859      1.491194
max        8.242258      6.115991


In [14]:
def dc_tau(home_goals, away_goals, lambda_home, lambda_away, rho):
    if home_goals == 0 and away_goals == 0:
        return max(1 - lambda_home * lambda_away * rho, 1e-6)
    elif home_goals == 1 and away_goals == 0:
        return max(1 + lambda_away * rho, 1e-6)
    elif home_goals == 0 and away_goals == 1:
        return max(1 + lambda_home * rho, 1e-6)
    elif home_goals == 1 and away_goals == 1:
        return max(1 - rho, 1e-6)
    else:
        return 1.0

def scoreline_probs(lambda_home, lambda_away, rho, max_goals=8):
    """Build full scoreline probability grid."""
    grid = np.zeros((max_goals + 1, max_goals + 1))
    for h in range(max_goals + 1):
        for a in range(max_goals + 1):
            grid[h, a] = (
                poisson.pmf(h, lambda_home) *
                poisson.pmf(a, lambda_away) *
                dc_tau(h, a, lambda_home, lambda_away, rho)
            )
    # Normalise so probs sum to 1
    grid /= grid.sum()
    return grid

In [15]:
def simulate_stats(rho, df, n_samples=1000):
    """
    For each match in df, sample n_samples scorelines using DC,
    return overall draw rate and avg goals/game.
    """
    draws = 0
    total_goals = 0
    total_matches = 0

    for _, row in df.iterrows():
        grid = scoreline_probs(row["pred_home"], row["pred_away"], rho)
        
        # Flatten grid into scoreline samples
        probs_flat = grid.flatten()
        indices = np.random.choice(len(probs_flat), size=n_samples, p=probs_flat)
        h_goals = indices // (grid.shape[1])
        a_goals = indices % (grid.shape[1])
        
        draws += (h_goals == a_goals).sum()
        total_goals += (h_goals + a_goals).sum()
        total_matches += n_samples

    draw_rate = draws / total_matches
    avg_goals = total_goals / total_matches
    return draw_rate, avg_goals

In [16]:
rho_candidates = np.linspace(-0.3, 0.0, 31)
results_grid = []

for rho in rho_candidates:
    draw_rate, avg_goals = simulate_stats(rho, calib_features, n_samples=200)
    results_grid.append({
        "rho": round(rho, 3),
        "draw_rate": round(draw_rate, 4),
        "avg_goals": round(avg_goals, 4),
    })
    print(f"rho={rho:.3f} | draw_rate={draw_rate:.3f} | avg_goals={avg_goals:.3f}")

grid_df = pd.DataFrame(results_grid)

rho=-0.300 | draw_rate=0.265 | avg_goals=2.824
rho=-0.290 | draw_rate=0.264 | avg_goals=2.825
rho=-0.280 | draw_rate=0.261 | avg_goals=2.826
rho=-0.270 | draw_rate=0.259 | avg_goals=2.826


KeyboardInterrupt: 

In [17]:
calib_features = calib_sorted[
    calib_sorted["date"] >= "1990-01-01"
][["home_elo", "away_elo", "elo_diff", "is_worldcup", "home_score", "away_score"]].copy()

print(calib_features.shape)
print(calib_features[["home_score", "away_score"]].mean())

(15718, 6)
home_score    1.747805
away_score    1.150846
dtype: float64


In [18]:
calib_wc = calib_sorted[
    calib_sorted["tournament"].str.contains("World Cup", na=False) &
    (calib_sorted["date"] >= "1990-01-01")
][["home_elo", "away_elo", "elo_diff", "is_worldcup", "home_score", "away_score"]].copy()

print(calib_wc.shape)
print(calib_wc[["home_score", "away_score"]].mean())

(5694, 6)
home_score    1.762908
away_score    1.122761
dtype: float64


In [23]:
# Revised targets based on actual data
target_draw = 0.265   # keep this, draw rate is fine
target_goals = 2.85   # match what's actually in the data
feature_cols = ["home_elo", "away_elo", "elo_diff", "is_worldcup"]

calib_wc = calib_wc.copy()  # avoid SettingWithCopyWarning
calib_wc["pred_home"] = np.clip(home_model.predict(calib_wc[feature_cols]), 0.1, None)
calib_wc["pred_away"] = np.clip(away_model.predict(calib_wc[feature_cols]), 0.1, None)

calib_sample = calib_wc.sample(n=500, random_state=42).reset_index(drop=True)

In [24]:
calib_sample = calib_wc.sample(n=500, random_state=42).reset_index(drop=True)

rho_candidates = np.linspace(-0.3, 0.0, 31)
results_grid = []

for rho in rho_candidates:
    draw_rate, avg_goals = simulate_stats(rho, calib_sample, n_samples=200)
    results_grid.append({
        "rho": round(rho, 3),
        "draw_rate": round(draw_rate, 4),
        "avg_goals": round(avg_goals, 4),
    })
    print(f"rho={rho:.3f} | draw_rate={draw_rate:.3f} | avg_goals={avg_goals:.3f}")

grid_df = pd.DataFrame(results_grid)

rho=-0.300 | draw_rate=0.254 | avg_goals=2.912
rho=-0.290 | draw_rate=0.252 | avg_goals=2.904
rho=-0.280 | draw_rate=0.248 | avg_goals=2.913
rho=-0.270 | draw_rate=0.246 | avg_goals=2.914
rho=-0.260 | draw_rate=0.245 | avg_goals=2.904
rho=-0.250 | draw_rate=0.245 | avg_goals=2.905
rho=-0.240 | draw_rate=0.243 | avg_goals=2.907
rho=-0.230 | draw_rate=0.240 | avg_goals=2.905
rho=-0.220 | draw_rate=0.240 | avg_goals=2.912
rho=-0.210 | draw_rate=0.236 | avg_goals=2.904
rho=-0.200 | draw_rate=0.237 | avg_goals=2.904
rho=-0.190 | draw_rate=0.232 | avg_goals=2.915
rho=-0.180 | draw_rate=0.234 | avg_goals=2.907
rho=-0.170 | draw_rate=0.230 | avg_goals=2.915
rho=-0.160 | draw_rate=0.228 | avg_goals=2.912
rho=-0.150 | draw_rate=0.229 | avg_goals=2.907
rho=-0.140 | draw_rate=0.225 | avg_goals=2.903
rho=-0.130 | draw_rate=0.221 | avg_goals=2.906
rho=-0.120 | draw_rate=0.220 | avg_goals=2.909
rho=-0.110 | draw_rate=0.217 | avg_goals=2.908
rho=-0.100 | draw_rate=0.218 | avg_goals=2.913
rho=-0.090 | 

In [25]:
# Target: draw rate 25-28%, avg goals 2.4-2.6
target_draw = 0.265  # midpoint of 25-28%
target_goals = 2.5   # midpoint of 2.4-2.6

grid_df["score"] = (
    abs(grid_df["draw_rate"] - target_draw) +
    abs(grid_df["avg_goals"] - target_goals) * 0.5
)

best = grid_df.loc[grid_df["score"].idxmin()]
DIXON_COLES_RHO = best["rho"]

print(f"Best rho: {DIXON_COLES_RHO}")
print(f"Draw rate: {best['draw_rate']:.3f}")
print(f"Avg goals: {best['avg_goals']:.3f}")

Best rho: -0.29
Draw rate: 0.252
Avg goals: 2.904


In [26]:
WC2022_START = "2022-11-20"
WC2022_END   = "2022-12-18"

wc2022 = results[
    (results["date"] >= WC2022_START) &
    (results["date"] <= WC2022_END)
].copy()

wc2022 = wc2022.sort_values("date").reset_index(drop=True)

# Home Elo
home_lookup = team_elo_df.rename(columns={"team": "home_team", "elo": "home_elo"})
wc2022 = pd.merge_asof(wc2022, home_lookup, on="date", by="home_team")

# Away Elo
away_lookup = team_elo_df.rename(columns={"team": "away_team", "elo": "away_elo"})
wc2022 = pd.merge_asof(wc2022, away_lookup, on="date", by="away_team")

wc2022["elo_diff"] = wc2022["home_elo"] - wc2022["away_elo"]
wc2022["is_worldcup"] = 1
wc2022[["home_elo", "away_elo"]] = wc2022[["home_elo", "away_elo"]].fillna(1500)

wc2022["pred_home"] = np.clip(home_model.predict(wc2022[feature_cols]), 0.1, None)
wc2022["pred_away"] = np.clip(away_model.predict(wc2022[feature_cols]), 0.1, None)

draw_rate_val, avg_goals_val = simulate_stats(DIXON_COLES_RHO, wc2022, n_samples=500)

print(f"WC 2022 sanity check:")
print(f"  Draw rate: {draw_rate_val:.3f}  (target ~0.25)")
print(f"  Avg goals: {avg_goals_val:.3f}  (target ~2.85)")

# Also check actual WC 2022 stats for comparison
actual_draws = (wc2022["home_score"] == wc2022["away_score"]).mean()
actual_goals = (wc2022["home_score"] + wc2022["away_score"]).mean()
print(f"\nActual WC 2022:")
print(f"  Draw rate: {actual_draws:.3f}")
print(f"  Avg goals: {actual_goals:.3f}")

WC 2022 sanity check:
  Draw rate: 0.280  (target ~0.25)
  Avg goals: 2.759  (target ~2.85)

Actual WC 2022:
  Draw rate: 0.241
  Avg goals: 2.598


In [27]:
dc_params = {"rho": float(DIXON_COLES_RHO)}

with open("../models/dixon_coles_params.json", "w") as f:
    json.dump(dc_params, f, indent=2)

print(f"Saved: dixon_coles_params.json")
print(dc_params)

Saved: dixon_coles_params.json
{'rho': -0.29}
